# Speech Emotion Recognition — CNN + Sequential & Attention Models

**Notebook 3 of 3** in the RAVDESS series.

| Notebook | Focus |
|----------|-------|
| 1 | Classical ML on hand-crafted features (MFCCs, Chroma, Spectral) |
| 2 | CNN on mel-spectrograms — regularization, augmentation, transfer learning |
| **3** | **CNN + sequential and attention modules — BiLSTM, Attention Pooling, Multi-Head Self-Attention** |

### Research question

> Can explicitly modelling **temporal dependencies** in mel-spectrograms improve speech emotion recognition beyond a plain CNN?

A plain CNN reads a mel-spectrogram like an image and captures **local** time-frequency patterns, but emotion also lives in **how** patterns change over time (rising pitch, trailing-off energy, rhythmic stress). This notebook adds sequence-modelling layers on top of the same CNN feature extractor and measures whether that temporal context helps.

### Experiments

| # | Architecture | Core idea |
|---|-------------|-----------|
| 1 | CNN + BiLSTM | CNN extracts local features → BiLSTM models temporal sequences |
| 2 | CNN + BiLSTM + Attention Pooling | Learn *which* time frames matter most |
| 3 | CNN + Multi-Head Self-Attention | Global frame interactions, no recurrence |
| 4 | CNN + BiLSTM + Multi-Head Attention | Full hybrid: local + sequential + global |

### Data pipeline

Identical to Notebook 2 — same preprocessing, same 70/15/15 split, same normalization. This means results are directly comparable across all three notebooks in the series.

> **Hardware:** Kaggle P100 GPU. The PyTorch reinstall cell below fixes a known CUDA/driver mismatch on that runtime.

### Code organization

1. Imports
2. Load Dataset
3. Feature Extraction — Mel Spectrogram, Normalization
4. Dataset — `AudioDataset`, `AugmentedAudioDataset`
5. Utilities — `train_model()`, `evaluate()`, `plot_history()`, `plot_confusion()`
6. Base CNN Feature Extractor
7. Experiment 1 — CNN + BiLSTM
8. Experiment 2 — CNN + BiLSTM + Attention Pooling
9. Experiment 3 — CNN + Multi-Head Self-Attention
10. Experiment 4 — CNN + BiLSTM + Multi-Head Attention
11. Comparison Table
12. Discussion

The design starts from **one reusable `CNNFeatureExtractor`** rather than rewriting the convolutional layers in every experiment. Each model then becomes a thin, clean subclass, and every experiment changes only the sequence-modelling component on top of it — which keeps the comparison across experiments fair and makes the contribution of each architectural choice easy to isolate.

## 0. Environment Setup

Reinstalls a CUDA 12.6-matched PyTorch wheel — the fix for the P100 CUDA mismatch on Kaggle.
Comment this out if your session already has a working GPU build; it costs ~2 minutes per run.

In [ ]:
!pip uninstall -y torch torchvision torchaudio -q
!pip install torch==2.7.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126 -q

import os
import random
import warnings

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed: int = SEED):
    """Reproducibility across numpy / python / torch (CPU + CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 1. Data Loading & Exploration

In [ ]:
wav_files = []
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".wav"):
            wav_files.append(os.path.join(dirname, filename))

print(f"Total audio files found: {len(wav_files):,}")
for p in wav_files[:3]:
    print(" ", p)

In [ ]:
EMOTIONS_MAP = {
    "01": "neutral",  "02": "calm",     "03": "happy",
    "04": "sad",      "05": "angry",    "06": "fearful",
    "07": "disgust",  "08": "surprised"
}

def get_emotion(path: str) -> str:
    code = os.path.basename(path).split("-")[2]
    return EMOTIONS_MAP.get(code, "unknown")

df = pd.DataFrame({"path": wav_files, "emotion": [get_emotion(p) for p in wav_files]})
n_before = len(df)
df = df[df["emotion"] != "unknown"].reset_index(drop=True)

if len(df) < n_before:
    print(f"Dropped {n_before - len(df)} files with unrecognized emotion codes")

print(f"Dataset shape: {df.shape}")

In [ ]:
label_counts = df["emotion"].value_counts().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(label_counts.index, label_counts.values,
             color=sns.color_palette("muted", len(label_counts)))
axes[0].set_xlabel("Number of Samples")
axes[0].set_title("Sample Count per Emotion Class")
for i, v in enumerate(label_counts.values):
    axes[0].text(v + 0.5, i, str(v), va="center", fontsize=10)

axes[1].pie(label_counts.values, labels=label_counts.index,
            autopct="%1.1f%%", startangle=140,
            colors=sns.color_palette("muted", len(label_counts)))
axes[1].set_title("Emotion Class Proportions")

plt.suptitle("Class Distribution — RAVDESS Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Feature Engineering

Identical to Notebook 2 — same sample rate, duration, mel parameters. This ensures the spectrograms
fed into these new architectures are the same as those used in all previous experiments.

In [ ]:
durations = np.array([
    librosa.get_duration(path=p) for p in tqdm(df["path"], desc="Measuring durations")
])
print(f"Min: {durations.min():.2f}s | Mean: {durations.mean():.2f}s | "
      f"Max: {durations.max():.2f}s | 95th pct: {np.percentile(durations, 95):.2f}s")

plt.figure(figsize=(8, 3))
plt.hist(durations, bins=50)
plt.xlabel("Duration (seconds)")
plt.ylabel("Count")
plt.title("Audio Length Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# ── Identical hyperparameters to Notebook 2 ──────────────────────────────────
SR          = 22050
N_MELS      = 128
DURATION    = 4          # seconds
SAMPLES     = SR * DURATION
N_FFT       = 2048
HOP_LENGTH  = 512

def extract_mel(path: str) -> np.ndarray:
    """Load, pad/truncate to fixed length, return log-mel spectrogram (n_mels, n_frames)."""
    signal, sr = librosa.load(path, sr=SR)
    if len(signal) < SAMPLES:
        signal = np.pad(signal, (0, SAMPLES - len(signal)))
    else:
        signal = signal[:SAMPLES]
    mel    = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=N_MELS,
                                             n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db

In [ ]:
X, y = [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting mel-spectrograms"):
    X.append(extract_mel(row["path"]))
    y.append(row["emotion"])

X = np.array(X, dtype=np.float32)
y = np.array(y)

print(f"X shape : {X.shape}  (n_samples, n_mels, n_frames)")
print(f"X range : [{X.min():.2f}, {X.max():.2f}] dB")

encoder = LabelEncoder()
y = encoder.fit_transform(y)
print(f"Classes : {list(encoder.classes_)}")

## 3. Shared Train / Val / Test Split & Normalization

**Identical to Notebook 2:** stratified 70/15/15 split, z-score normalization using train-set
statistics only. Keeping this the same means every number in this notebook is directly
comparable to results from Notebooks 1 and 2.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
MEL_MEAN = X_train.mean()
MEL_STD  = X_train.std()

def normalize(x):
    return (x - MEL_MEAN) / (MEL_STD + 1e-8)

X_train_norm = normalize(X_train)
X_val_norm   = normalize(X_val)
X_test_norm  = normalize(X_test)

print(f"Normalization — mean: {MEL_MEAN:.3f}  std: {MEL_STD:.3f}")
print(f"Normalized train range: [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]")

## 4. Dataset Classes & Augmentation

Same `AudioDataset`, `AugmentedAudioDataset`, `spec_augment`, `mixup_batch`, and `soft_ce_loss`
as Notebook 2 — no changes. All four experiments in this notebook use **SpecAugment + Mixup** on
the training data, since Notebook 2 showed augmentation consistently helps.

In [ ]:
class AudioDataset(Dataset):
    """Plain dataset — shape (1, n_mels, n_frames)."""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)
    def __len__(self):  return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


def spec_augment(spec: torch.Tensor,
                 n_time_masks=2, n_freq_masks=2,
                 time_mask_pct=0.10, freq_mask_pct=0.10) -> torch.Tensor:
    """SpecAugment (Park et al., 2019) — zero out random time and frequency bands."""
    spec = spec.clone()
    _, n_mels, n_frames = spec.shape

    for _ in range(n_freq_masks):
        w  = max(1, int(n_mels * freq_mask_pct))
        f0 = random.randint(0, max(0, n_mels - w))
        spec[:, f0:f0 + w, :] = 0.0

    for _ in range(n_time_masks):
        w  = max(1, int(n_frames * time_mask_pct))
        t0 = random.randint(0, max(0, n_frames - w))
        spec[:, :, t0:t0 + w] = 0.0
    return spec


class AugmentedAudioDataset(Dataset):
    """AudioDataset + on-the-fly SpecAugment (train=True only)."""
    def __init__(self, X, y, train: bool = True):
        self.X     = torch.FloatTensor(X).unsqueeze(1)
        self.y     = torch.LongTensor(y)
        self.train = train
    def __len__(self):  return len(self.X)
    def __getitem__(self, idx):
        x = self.X[idx]
        if self.train:
            x = spec_augment(x)
        return x, self.y[idx]


def mixup_batch(x, y, num_classes, alpha=0.2):
    """Mixup (Zhang et al., 2018) — blend pairs of inputs and soft-label targets."""
    lam  = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    perm = torch.randperm(x.size(0), device=x.device)
    x_mixed  = lam * x + (1 - lam) * x[perm]
    y_onehot = F.one_hot(y, num_classes=num_classes).float()
    y_mixed  = lam * y_onehot + (1 - lam) * y_onehot[perm]
    return x_mixed, y_mixed


def soft_ce_loss(logits, soft_targets):
    """Cross-entropy with soft (Mixup) targets."""
    return -(soft_targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

In [ ]:
# Sanity check — visualize one normalized spectrogram
x_sample, y_sample = AudioDataset(X_train_norm, y_train)[104]
plt.figure(figsize=(10, 4))
librosa.display.specshow(x_sample[0].numpy(), sr=SR, hop_length=HOP_LENGTH,
                         x_axis="time", y_axis="mel")
plt.colorbar(format="%+2.1f")
plt.title(f"Normalized log-mel spectrogram | label = {encoder.classes_[y_sample]}")
plt.tight_layout()
plt.show()

## 5. Shared Training Utilities

A reusable training loop (with early stopping + LR scheduling) and evaluation function, so every
experiment calls the same code instead of duplicating the loop 4 times. We keep a `results`
dictionary to collect metrics for the final comparison.

In [ ]:
results = {}  # populated by each experiment: results[name] = {"test_acc": ..., "train_acc": ..., ...}


def train_model(model, train_loader, val_loader, optimizer, scheduler=None,
                 criterion=None, num_epochs=200, patience=10, use_mixup=False,
                 num_classes=None, model_name="model", verbose_every=5):
    """
    Generic training loop with early stopping on validation loss.
    If use_mixup=True, applies mixup to each training batch and uses a soft-label
    cross-entropy; validation always uses plain (unmixed) data and standard CE.
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    counter = 0
    history = {"train_loss": [], "val_loss": []}
    ckpt_path = f"/kaggle/working/{model_name}_best.pth"

    for epoch in range(num_epochs):
        # ---- Train ----
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()

            if use_mixup:
                X_mixed, y_mixed = mixup_batch(X_batch, y_batch, num_classes)
                outputs = model(X_mixed)
                loss = soft_ce_loss(outputs, y_mixed)
            else:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # ---- Validate (always on clean, unaugmented data) ----
        model.eval()
        running_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                running_loss += loss.item()
        val_loss = running_loss / len(val_loader)

        if scheduler is not None:
            scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if (epoch + 1) % verbose_every == 0 or epoch == 0:
            lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch+1:03d} | Train Loss={train_loss:.4f} | "
                  f"Val Loss={val_loss:.4f} | LR={lr:.6f}")

        # ---- Early stopping ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch+1} (best val loss={best_val_loss:.4f})")
                break

    model.load_state_dict(torch.load(ckpt_path))
    return model, history


@torch.no_grad()
def evaluate(model, loader):
    """Run inference over a loader and return (accuracy, y_true, y_pred)."""
    model.eval()
    y_true, y_pred = [], []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(y_batch.numpy())
        y_pred.extend(predicted.cpu().numpy())
    return accuracy_score(y_true, y_pred), np.array(y_true), np.array(y_pred)


def plot_history(history, title):
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=encoder.classes_, yticklabels=encoder.classes_)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

## 6. Base CNN Feature Extractor

One reusable convolutional trunk shared by all four experiments. It reads the mel-spectrogram like
an image and reduces it to a **sequence** of local time-frequency features — a stack of
`Conv2d → BatchNorm2d → ReLU → MaxPool2d` blocks that shrink the frequency axis while preserving
enough of the time axis to hand off to a sequence model. Every experiment below plugs a different
sequence-modelling head on top of this same trunk, so any accuracy difference between experiments
can be attributed to the head, not to different features.

In [ ]:
class CNNFeatureExtractor(nn.Module):
    """
    Extract local time-frequency features from a mel spectrogram.

    Input:
        (B, 1, 128, 173)

    Output:
        (B, T, C)
        where T = sequence length
              C = feature dimension
    """

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2,1)),

            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2,1)),

            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.10)
        )

    def forward(self, x):

        # Input
        # (B, 1, 128, 173)

        x = self.features(x)

        # After CNN
        # (B, 32, 8, 43)

        # Average over frequency axis
        x = x.mean(dim=2)

        # (B, 32, 43)

        # Convert to sequence format
        x = x.transpose(1, 2)

        # (B, 43, 32)
        # 43 time steps
        # 32 features per frame

        return x

## 7. Experiment 1 — CNN + BiLSTM

**Idea:** stop reading the spectrogram frame-by-frame in isolation. After the CNN produces a
sequence of local features, feed that sequence through a **Bidirectional LSTM** so each time step's
representation is informed by everything before *and* after it in the clip.

```
Mel Spectrogram
       │
       ▼
      CNN
(Local Features)
       │
       ▼
Feature Sequence
       │
       ▼
    BiLSTM
(Temporal Modeling)
       │
       ▼
  Classifier
```

In [ ]:
class CNNBiLSTM(nn.Module):
    """
    CNN + Bidirectional LSTM for Speech Emotion Recognition.

    Input:
        (B,1,128,173)

    Output:
        (B,num_classes)
    """

    def __init__(
        self,
        num_classes,
        hidden_size=128,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = CNNFeatureExtractor()

        # Project CNN features
        self.projection = nn.Linear(32, 128)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        # ---------------------------
        # CNN
        # (B,1,128,173)
        # ->
        # (B,43,32)
        # ---------------------------
        x = self.cnn(x)

        # ---------------------------
        # Projection
        # (B,43,128)
        # ---------------------------
        x = self.projection(x)

        # ---------------------------
        # BiLSTM
        # (B,43,256)
        # ---------------------------
        x, _ = self.lstm(x)

        # ---------------------------
        # Global temporal average
        # (B,256)
        # ---------------------------
        x = x.mean(dim=1)

        return self.classifier(x)

In [ ]:
set_seed(SEED)

train_dataset = AugmentedAudioDataset(
    X_train_norm,
    y_train,
    train=True
)

val_dataset = AudioDataset(
    X_val_norm,
    y_val
)

test_dataset = AudioDataset(
    X_test_norm,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

clean_train_loader = DataLoader(
    AudioDataset(X_train_norm, y_train),
    batch_size=32
)

In [ ]:
model = CNNBiLSTM(
    num_classes=len(encoder.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

model, history = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion=criterion,
    num_epochs=200,
    patience=25,
    use_mixup=True,
    num_classes=len(encoder.classes_),
    model_name="cnn_bilstm"
)

plot_history(history, "CNN + BiLSTM")

In [ ]:
test_acc, yt, yp = evaluate(model, test_loader)
train_acc, _, _ = evaluate(model, clean_train_loader)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Gap           : {train_acc-test_acc:.4f}")

print(classification_report(yt, yp, target_names=encoder.classes_))

plot_confusion(yt, yp, "CNN + BiLSTM")

results["CNN + BiLSTM"] = {"train_acc": train_acc, "test_acc": test_acc, "gap": train_acc - test_acc}

## 8. Experiment 2 — CNN + BiLSTM + Attention Pooling

Experiment 1 collapses the BiLSTM output with a plain average over time — every frame counts
equally. Here we replace that average with a small **attention pooling** head that *learns which
time frames matter most* (e.g. the syllable where anger peaks) instead of treating all frames as
equally important.

```
   CNN
    ↓
  BiLSTM
    ↓
Attention Pooling
    ↓
 Classifier
```

This is still **not** a Transformer — the attention here is only used to aggregate the BiLSTM
outputs into a single vector, not to let frames attend to each other.

In [ ]:
class AttentionPooling(nn.Module):
    """
    Learns a scalar importance score per time step and pools the sequence
    into a single vector as a weighted sum (instead of a plain average).

    Input:
        (B, T, D)

    Output:
        pooled : (B, D)
        weights: (B, T)  — attention weight per time step, for inspection
    """
    def __init__(self, input_dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or input_dim // 2
        self.attn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        scores  = self.attn(x)                    # (B, T, 1)
        weights = torch.softmax(scores, dim=1)     # (B, T, 1)
        pooled  = (x * weights).sum(dim=1)         # (B, D)
        return pooled, weights.squeeze(-1)


class CNNBiLSTMAttention(nn.Module):
    """
    CNN + BiLSTM + Attention Pooling for Speech Emotion Recognition.

    Input:
        (B,1,128,173)

    Output:
        (B,num_classes)
    """

    def __init__(
        self,
        num_classes,
        hidden_size=128,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = CNNFeatureExtractor()
        self.projection = nn.Linear(32, 128)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.attn_pool = AttentionPooling(hidden_size * 2)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x, return_attn=False):
        x = self.cnn(x)               # (B,43,32)
        x = self.projection(x)        # (B,43,128)
        x, _ = self.lstm(x)           # (B,43,256)
        pooled, attn_weights = self.attn_pool(x)  # (B,256), (B,43)
        out = self.classifier(pooled)
        if return_attn:
            return out, attn_weights
        return out

In [ ]:
set_seed(SEED)

model = CNNBiLSTMAttention(
    num_classes=len(encoder.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

model, history = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion=criterion,
    num_epochs=200,
    patience=25,
    use_mixup=True,
    num_classes=len(encoder.classes_),
    model_name="cnn_bilstm_attn"
)

plot_history(history, "CNN + BiLSTM + Attention Pooling")

In [ ]:
test_acc, yt, yp = evaluate(model, test_loader)
train_acc, _, _ = evaluate(model, clean_train_loader)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Gap           : {train_acc-test_acc:.4f}")

print(classification_report(yt, yp, target_names=encoder.classes_))

plot_confusion(yt, yp, "CNN + BiLSTM + Attention Pooling")

results["CNN + BiLSTM + Attention"] = {"train_acc": train_acc, "test_acc": test_acc, "gap": train_acc - test_acc}

## 9. Experiment 3 — CNN + Multi-Head Self-Attention

Now the BiLSTM is removed entirely. Instead, every time frame attends to **every other** frame
directly through **Multi-Head Self-Attention** — no recurrence at all. Because attention has no
built-in notion of order, a fixed **sinusoidal positional encoding** is added to the CNN feature
sequence first so the model can still tell "early in the clip" from "late in the clip".

```
     CNN
      ↓
MultiHead Self-Attention
      ↓
  Classifier
```

This is basically the core idea behind Transformers, applied to a short feature sequence instead
of a sentence of tokens.

In [ ]:
class PositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding (Vaswani et al., 2017)."""
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x: (B, T, D)
        return x + self.pe[:, :x.size(1)]


class CNNMHAttention(nn.Module):
    """
    CNN + Multi-Head Self-Attention for Speech Emotion Recognition (no recurrence).

    Input:
        (B,1,128,173)

    Output:
        (B,num_classes)
    """

    def __init__(
        self,
        num_classes,
        d_model=128,
        n_heads=4,
        num_attn_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = CNNFeatureExtractor()
        self.projection = nn.Linear(32, d_model)
        self.pos_enc = PositionalEncoding(d_model)

        self.attn_layers = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                   dropout=dropout, batch_first=True)
            for _ in range(num_attn_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_attn_layers)])
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)                 # (B,43,32)
        x = self.projection(x)          # (B,43,128)
        x = self.pos_enc(x)             # (B,43,128) + position

        for attn, norm in zip(self.attn_layers, self.norms):
            attn_out, _ = attn(x, x, x)          # every frame attends to every frame
            x = norm(x + self.dropout(attn_out)) # residual + layer norm

        x = x.mean(dim=1)               # (B,128) global average over time
        return self.classifier(x)

In [ ]:
set_seed(SEED)

model = CNNMHAttention(
    num_classes=len(encoder.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

model, history = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion=criterion,
    num_epochs=200,
    patience=25,
    use_mixup=True,
    num_classes=len(encoder.classes_),
    model_name="cnn_mha"
)

plot_history(history, "CNN + Multi-Head Self-Attention")

In [ ]:
test_acc, yt, yp = evaluate(model, test_loader)
train_acc, _, _ = evaluate(model, clean_train_loader)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Gap           : {train_acc-test_acc:.4f}")

print(classification_report(yt, yp, target_names=encoder.classes_))

plot_confusion(yt, yp, "CNN + Multi-Head Self-Attention")

results["CNN + MultiHead Attention"] = {"train_acc": train_acc, "test_acc": test_acc, "gap": train_acc - test_acc}

## 10. Experiment 4 — CNN + BiLSTM + Multi-Head Attention

The strongest hybrid: combine all three ideas at once — **local patterns** (CNN),
**sequential modeling** (BiLSTM), and **global relationships** (Multi-Head Attention).

```
   CNN
    ↓
  BiLSTM
    ↓
MultiHead Attention
    ↓
 Classifier
```

Many SER papers still use exactly this architecture, since the BiLSTM gives the attention layer
already order-aware representations to work with (so no positional encoding is needed here).

In [ ]:
class CNNBiLSTMMHAttention(nn.Module):
    """
    CNN + BiLSTM + Multi-Head Attention for Speech Emotion Recognition.

    Input:
        (B,1,128,173)

    Output:
        (B,num_classes)
    """

    def __init__(
        self,
        num_classes,
        hidden_size=128,
        num_layers=2,
        n_heads=4,
        dropout=0.3
    ):
        super().__init__()

        self.cnn = CNNFeatureExtractor()
        self.projection = nn.Linear(32, 128)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.mha = nn.MultiheadAttention(
            embed_dim=hidden_size * 2,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)                       # (B,43,32)
        x = self.projection(x)                # (B,43,128)
        x, _ = self.lstm(x)                    # (B,43,256)

        attn_out, _ = self.mha(x, x, x)        # global frame interactions
        x = self.norm(x + self.dropout(attn_out))

        x = x.mean(dim=1)                      # (B,256)
        return self.classifier(x)

In [ ]:
set_seed(SEED)

model = CNNBiLSTMMHAttention(
    num_classes=len(encoder.classes_)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

model, history = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion=criterion,
    num_epochs=200,
    patience=25,
    use_mixup=True,
    num_classes=len(encoder.classes_),
    model_name="cnn_bilstm_mha"
)

plot_history(history, "CNN + BiLSTM + Multi-Head Attention")

In [ ]:
test_acc, yt, yp = evaluate(model, test_loader)
train_acc, _, _ = evaluate(model, clean_train_loader)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Gap           : {train_acc-test_acc:.4f}")

print(classification_report(yt, yp, target_names=encoder.classes_))

plot_confusion(yt, yp, "CNN + BiLSTM + Multi-Head Attention")

results["CNN + BiLSTM + MultiHead Attention"] = {"train_acc": train_acc, "test_acc": test_acc, "gap": train_acc - test_acc}

## 11. Comparison Table

All four experiments share the same data pipeline, augmentation, training loop, and optimizer
settings — so the differences below isolate the effect of the sequence-modelling head.

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values("test_acc", ascending=False)
comparison_df = comparison_df.rename(columns={
    "train_acc": "Train Acc",
    "test_acc": "Test Acc",
    "gap": "Train-Test Gap"
})
comparison_df

In [ ]:
plt.figure(figsize=(9, 4.5))
order = comparison_df.index
bars = plt.bar(order, comparison_df["Test Acc"], color=sns.color_palette("muted", len(order)))
plt.ylabel("Test Accuracy")
plt.title("Experiment Comparison — Test Accuracy")
plt.xticks(rotation=15, ha="right")
for bar, val in zip(bars, comparison_df["Test Acc"]):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.005, f"{val:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 12. Discussion

The goal of this notebook was not to beat ResNet18 — it was to answer a research question:

> Can explicitly modeling temporal dependencies improve CNN-based speech emotion recognition?

Each experiment introduces exactly **one** new concept on top of the last, so the comparison table
above can be read as a controlled ablation:

| Experiment | New idea |
|---|---|
| CNN (baseline, Notebook 2) | Local feature extraction |
| CNN + BiLSTM | Temporal modeling |
| CNN + BiLSTM + Attention | Learn which frames matter |
| CNN + Multi-Head Attention | Global interactions, no recurrence |
| CNN + BiLSTM + Multi-Head Attention | Local + sequential + global, combined |

**Data pipeline recap:** audio → log-mel spectrogram → normalization → `DataLoader` →
CNN + sequence model, with no changes across experiments — the same pipeline as Notebook 2 — so
every number above is directly comparable across all three notebooks in the series.

**Design note:** every model here is a thin subclass built on one shared `CNNFeatureExtractor`,
rather than four independent copies of the convolutional stack. That keeps the notebook
maintainable and, more importantly, keeps the comparison fair: only the sequence-modelling
component changes between experiments, so any accuracy delta can be attributed to *that* choice
rather than to incidental differences in the CNN.

By the end of this notebook, the natural conclusion is a preview of why Transformers were
invented in the first place: once you've built attention purely to *aggregate* a sequence
(Experiment 2), and then noticed it also works to let a sequence *interact with itself* without
any recurrence at all (Experiment 3), the hybrid in Experiment 4 is the bridge between the
RNN-based era of sequence modeling and the attention-only era that followed it.